# NYC yellow taxi: rain, and weekend nights

Two questions against `hourly_trip_activity`, March–April 2025 — the months the
streaming demo replays.

1. Does rain change the number of completed trips? **Yes, +18.2%.**
2. How do weekend nights compare with Monday mornings? **Shorter rides, more of them,
   more revenue per hour.**

These are completed trips, not demand: someone who wanted a cab and did not get one is not
here. Revenue is the fare charged, not profit — there are no costs in this data.

In [ ]:
# Local setup. `make streaming-demo-fast` builds the marts; this reads them.
import os, sys
from pathlib import Path

# Jupyter starts in the notebook's directory, and the warehouse path is relative to the root.
if not Path("pyproject.toml").exists() and Path("../pyproject.toml").exists():
    os.chdir("..")
sys.path.insert(0, "src")

from pipeline.spark import local_session

spark = local_session("taxi-analysis")
spark.sql(f"USE {os.environ.get('DBT_SCHEMA', 'pipeline')}")


def q(sql):
    """Run a query and hand back a pandas frame, which renders as a table."""
    return spark.sql(sql).toPandas()


# Part 1 — Rain

## Coverage

Hours where the station reported no rainfall reading are left out. They are never counted
as dry.

In [ ]:
q("""
select
  weather_station,
  count(*)                                                        as station_hours,
  count_if(precip_bucket is null)                                 as hours_without_reading,
  sum(trips)                                                      as trips,
  round(100.0 * count_if(precip_bucket is not null) / count(*), 1) as pct_hours_usable
from hourly_trip_activity
group by weather_station
order by trips desc
""")


## Naive vs like-for-like

Naive compares every wet hour to every dry hour. If rain happened to fall mostly at rush
hour, that number would really be showing rush hour.

Like-for-like puts hours into groups: same station, same hour of day, both weekdays or both
weekends. So 8am is only ever compared with 8am. Busy groups count for more than quiet ones
when the groups are added up.

In [ ]:
q("""
with hour_groups as (
  select
    weather_station,
    hour(utc_hour)                as hour_of_day,
    dayofweek(utc_hour) in (1, 7) as is_weekend,
    avg(case when precip_bucket = 'measurable' then trips end) as wet_mean,
    avg(case when precip_bucket = 'dry'        then trips end) as dry_mean,
    sum(case when precip_bucket = 'dry'        then trips end) as weight
  from hourly_trip_activity
  group by 1, 2, 3
)
select 'naive' as method,
       (select round(100 * (avg(case when precip_bucket = 'measurable' then trips end)
                          / avg(case when precip_bucket = 'dry' then trips end) - 1), 1)
        from hourly_trip_activity) as pct_difference
union all
select 'like-for-like',
       round(100 * (sum(wet_mean / dry_mean * weight) / sum(weight) - 1), 1)
from hour_groups where wet_mean is not null and dry_mean is not null
""")


6.2% naive, 18.2% like-for-like. Close together, so time of day was not distorting the
answer much. Worth checking, not worth skipping.

## By station, rainfall amount, and month

If rain is really behind this, heavier rain should show a bigger difference.

Months check the answer holds through the window. Weeks do not work: one week has about 26
rainy hours split across about 72 groups, so most groups get one rainy hour or none. Week
numbers swing from -42% to +69%, which is sample size talking, not weather.

In [ ]:
q("""
with hour_groups as (
  select
    weather_station,
    date_format(utc_hour, 'yyyy-MM')  as month,
    hour(utc_hour)                    as hour_of_day,
    dayofweek(utc_hour) in (1, 7)     as is_weekend,
    avg(case when precip_bucket = 'measurable' then trips end)            as wet_mean,
    avg(case when precip_in >= 0.05 then trips end)                       as heavy_mean,
    avg(case when precip_bucket = 'measurable' and precip_in < 0.05 then trips end) as light_mean,
    avg(case when precip_bucket = 'dry'        then trips end)            as dry_mean,
    sum(case when precip_bucket = 'dry'        then trips end)            as weight
  from hourly_trip_activity
  group by 1, 2, 3, 4
)
select 'station: ' || weather_station as cut,
       round(100 * (sum(wet_mean / dry_mean * weight) / sum(weight) - 1), 1) as pct_difference
from hour_groups where wet_mean is not null and dry_mean is not null group by weather_station
union all
select 'month: ' || month,
       round(100 * (sum(wet_mean / dry_mean * weight) / sum(weight) - 1), 1)
from hour_groups where wet_mean is not null and dry_mean is not null group by month
union all
select 'rain: lighter (< 0.05 in/hr)',
       round(100 * (sum(light_mean / dry_mean * weight) / sum(weight) - 1), 1)
from hour_groups where light_mean is not null and dry_mean is not null
union all
select 'rain: heavier (>= 0.05 in/hr)',
       round(100 * (sum(heavy_mean / dry_mean * weight) / sum(weight) - 1), 1)
from hour_groups where heavy_mean is not null and dry_mean is not null
order by cut
""")


## Does the answer depend on my choices?

One choice changed per row. All land between +12% and +14%, so no single choice is holding
the answer up.

In [ ]:
q("""
with hour_groups as (
  select
    weather_station, hour(utc_hour) as hour_of_day, dayofweek(utc_hour) in (1, 7) as is_weekend,
    avg(case when precip_bucket = 'measurable'            then trips end) as wet_mean,
    avg(case when precip_bucket in ('measurable','trace') then trips end) as wet_incl_trace,
    avg(case when precip_bucket = 'dry'                   then trips end) as dry_mean,
    avg(case when precip_bucket in ('dry','trace')        then trips end) as dry_incl_trace,
    avg(case when precip_bucket = 'measurable' then movement_eligible_trips end) as wet_eligible,
    avg(case when precip_bucket = 'dry'        then movement_eligible_trips end) as dry_eligible,
    avg(case when precip_bucket = 'measurable' then total_revenue end) as wet_revenue,
    avg(case when precip_bucket = 'dry'        then total_revenue end) as dry_revenue,
    sum(case when precip_bucket = 'dry'        then trips end)         as weight
  from hourly_trip_activity
  group by 1, 2, 3
)
select 'headline' as variant,
       round(100 * (sum(wet_mean / dry_mean * weight) / sum(weight) - 1), 1) as pct_difference
from hour_groups where wet_mean is not null and dry_mean is not null
union all
select 'trace as wet', round(100 * (sum(wet_incl_trace / dry_mean * weight) / sum(weight) - 1), 1)
from hour_groups where wet_incl_trace is not null and dry_mean is not null
union all
select 'trace as dry', round(100 * (sum(wet_mean / dry_incl_trace * weight) / sum(weight) - 1), 1)
from hour_groups where wet_mean is not null and dry_incl_trace is not null
union all
select 'eligible trips only', round(100 * (sum(wet_eligible / dry_eligible * weight) / sum(weight) - 1), 1)
from hour_groups where wet_eligible is not null and dry_eligible is not null
union all
select 'revenue not trips', round(100 * (sum(wet_revenue / dry_revenue * weight) / sum(weight) - 1), 1)
from hour_groups where wet_revenue is not null and dry_revenue is not null
union all
select 'unweighted', round(100 * (avg(wet_mean / dry_mean) - 1), 1)
from hour_groups where wet_mean is not null and dry_mean is not null
""")


## Graph: rain effect by month

In [ ]:
import matplotlib.pyplot as plt

df = spark.sql("""
  with hour_groups as (
    select date_format(utc_hour,'yyyy-MM') as month, weather_station,
           hour(utc_hour) as hod, dayofweek(utc_hour) in (1,7) as we,
           avg(case when precip_bucket='measurable' then trips end) as wet,
           avg(case when precip_bucket='dry' then trips end) as dry,
           sum(case when precip_bucket='dry' then trips end) as w
    from hourly_trip_activity group by 1,2,3,4)
  select month, round(100*(sum(wet/dry*w)/sum(w)-1),1) as pct_difference
  from hour_groups where wet is not null and dry is not null group by month order by month
""").toPandas()

ax = df.plot.bar(x="month", y="pct_difference", legend=False, color="#2563eb")
ax.set_ylabel("% more trips in wet hours"); ax.set_xlabel("")
ax.axhline(0, color="grey", linewidth=1)
plt.show()


# Part 2 — Weekend nights vs Monday mornings

## Local time

The table stores UTC, because that is what lines trips up with weather. But "Friday night"
means New York time, so convert first. Cutting on UTC directly would file five hours of every
Friday night under Saturday.

The clocks went forward on 9 March. Below: 06:00 UTC is 01:00 in New York, 07:00 UTC is 03:00.
There was no 02:00 that day.

In [ ]:
q("""
select
  utc_hour,
  date_format(from_utc_timestamp(utc_hour, 'America/New_York'), 'EEEE HH:mm') as local_time
from hourly_trip_activity
where weather_station = 'central_park'
  and utc_hour between '2025-03-09 05:00:00' and '2025-03-09 09:00:00'
order by utc_hour
""")


## Windows

- Friday night — Fri 22:00 to Sat 03:59 local
- Saturday night — Sat 22:00 to Sun 03:59 local
- Monday morning — Mon 06:00 to 10:59 local

Nights run past midnight on purpose. Using the calendar date would cut each night in half.

In [ ]:
q("""
with windows as (
  select
    case
      when dayofweek(lh) = 6 and hour(lh) >= 22 then 'Friday night'
      when dayofweek(lh) = 7 and hour(lh) <= 3  then 'Friday night'
      when dayofweek(lh) = 7 and hour(lh) >= 22 then 'Saturday night'
      when dayofweek(lh) = 1 and hour(lh) <= 3  then 'Saturday night'
      when dayofweek(lh) = 2 and hour(lh) between 6 and 10 then 'Monday morning'
    end as period,
    trips, eligible, total_revenue, avg_distance_miles, avg_duration_minutes
  from (
    select from_utc_timestamp(utc_hour, 'America/New_York') as lh,
           trips, movement_eligible_trips as eligible, total_revenue,
           avg_distance_miles, avg_duration_minutes
    from hourly_trip_activity
  )
)
select
  period,
  sum(trips)                                                     as trips,
  round(sum(trips) / count(*), 0)                                as trips_per_station_hour,
  round(sum(avg_duration_minutes * eligible) / sum(eligible), 2)  as mean_minutes,
  round(sum(avg_distance_miles  * eligible) / sum(eligible), 2)   as mean_miles,
  round(sum(avg_distance_miles * eligible) / sum(avg_duration_minutes * eligible) * 60, 1) as mph,
  round(sum(total_revenue) / sum(trips), 2)                      as revenue_per_trip,
  round((sum(total_revenue) / sum(trips))
        / (sum(avg_duration_minutes * eligible) / sum(eligible)), 2) as revenue_per_ride_minute,
  round(sum(total_revenue) / count(*), 0)                        as revenue_per_station_hour
from windows
where period is not null
group by period
order by period
""")


Weekend night rides are about 2.5 minutes shorter, cover less ground, and move faster
(13 mph against 12.2). Monday morning traffic is the likely reason.

One Monday fare is worth more, $25.76 against $23.15, because the ride is longer. But every
minute of driving earns less, $1.54 against $1.65, and there are far fewer fares to pick up.
An hour of Saturday night brings in about 50% more than an hour of Monday morning.

So the better shift depends on which of the three numbers you care about.

## Graph: trips through the day

In [ ]:
import matplotlib.pyplot as plt

df = spark.sql("""
  select date_format(from_utc_timestamp(utc_hour,'America/New_York'),'EEEE') as day,
         hour(from_utc_timestamp(utc_hour,'America/New_York')) as local_hour,
         round(avg(trips),0) as mean_trips
  from hourly_trip_activity
  where date_format(from_utc_timestamp(utc_hour,'America/New_York'),'EEEE')
        in ('Friday','Saturday','Monday')
  group by 1,2 order by local_hour
""").toPandas()

ax = df.pivot(index="local_hour", columns="day", values="mean_trips").plot()
ax.set_ylabel("mean trips per station-hour"); ax.set_xlabel("hour of day (local)")
plt.show()


# Limitations

- Completed trips, not demand. A rider who found no cab leaves no trace.
- Revenue, not profit. No fuel, tolls, vehicle, or unpaid time between fares. Fare includes
  tips and surcharges, so it is not driver take-home either.
- Three weather stations stand in for five boroughs. Central Park covers Manhattan and 87% of
  trips, and has the worst record: 2.3% of its hours have no reading. A shower that hits
  Midtown but misses the park counts as dry, which makes the rain effect look smaller.
- Time and distance leave out trips with impossible values. Revenue counts every trip. The
  two sets differ by about 4.7%.
- March and April only. No summer storms, no snow.
- This shows things happening together, not one causing the other. The comparison takes out
  hour of day, day of week and station. Nothing else.